# M06.P03 — Retrieval evaluation

Código completo y ejecutable. El objetivo es ejecutar, inspeccionar, modificar y explicar.

Para mantener el laboratorio estable en CPU y evitar dependencias opcionales de visión/Transformers, este notebook usa un retriever TF-IDF local con scikit-learn. El objetivo pedagógico de P03 es evaluar retrieval, no comparar modelos de embeddings.


In [ ]:
from pathlib import Path
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

class TfidfStore:
    def __init__(self, documents):
        self.documents = list(documents)
        self.vectorizer = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2))
        self.matrix = self.vectorizer.fit_transform([d.page_content for d in self.documents])

    def similarity_search(self, query, k=4):
        query_vector = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vector, self.matrix)[0]
        ranking = np.argsort(scores)[::-1][:k]
        return [self.documents[i] for i in ranking]

def parse_markdown(path: Path) -> Document:
    text = path.read_text(encoding="utf-8")
    metadata = {"file": path.name}
    body = text
    if text.startswith("---"):
        parts = text.split("---", 2)
        if len(parts) == 3:
            raw_meta = parts[1]
            body = parts[2].strip()
            for line in raw_meta.splitlines():
                if ":" in line:
                    key, value = line.split(":", 1)
                    metadata[key.strip()] = value.strip().strip('\"')
    return Document(page_content=body, metadata=metadata)

def build_store(kb_path: Path):
    documents = [parse_markdown(p) for p in sorted(kb_path.glob("*.md"))]
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
    chunks = splitter.split_documents(documents)
    store = TfidfStore(chunks)
    return documents, chunks, store

def only_current(docs):
    return [d for d in docs if d.metadata.get("status") == "CURRENT"]

KB = Path("../assets/knowledge_base")
EVAL = Path("../assets/retrieval_eval.jsonl")
if not KB.exists():
    KB = Path("assets/knowledge_base")
    EVAL = Path("assets/retrieval_eval.jsonl")

_, _, store = build_store(KB)
cases = [json.loads(line) for line in EVAL.read_text(encoding="utf-8").splitlines() if line.strip()]

def evaluate_case(case, k):
    docs = only_current(store.similarity_search(case["question"], k=k))
    source_ids = [d.metadata.get("source_id") for d in docs]
    if case["expected_status"] == "NO_EVIDENCE":
        return {"id": case["id"], "expected": [], "retrieved": source_ids, "is_no_evidence_case": True, "hit": None}
    hit = any(s in case["expected_source_ids"] for s in source_ids)
    return {"id": case["id"], "expected": case["expected_source_ids"], "retrieved": source_ids, "is_no_evidence_case": False, "hit": hit}

def hit_rate_at_k(rows):
    positive = [r for r in rows if not r["is_no_evidence_case"]]
    return sum(bool(r["hit"]) for r in positive) / len(positive) if positive else 0.0

for k in [2, 4]:
    rows = [evaluate_case(case, k) for case in cases]
    print("\nk =", k)
    for row in rows:
        print(row)
    print("Hit Rate@", k, "=", hit_rate_at_k(rows))

print("\nDecisión recomendada: empezar con k=2 y aumentar solo si el eval set demuestra falta de cobertura.")


## Experimentos

1. Compara los resultados de `k=2` y `k=4`.
2. Revisa los casos `NO_EVIDENCE`.
3. Decide qué `k` llevarías a P04 y justifica la decisión.
